In [49]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import time
import math
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.svm import LinearSVC 

In [2]:
# вариант 1 (Мишин генератор)
data = np.loadtxt('data_model/linData_1000_10_09.txt')
Y_full = data[:, 0]
X_full = data[:, 1:]


In [2]:
# вариант 2 (генератор DataGeneration2)
#data = np.load('data_model/dg2_lin_1000_10_02_08.npz')
data = np.load('data_model/dg2_lin_100k_1k_b0_01_08.npz')
#data = np.load('data_model/dg2_lin_20_10_02_08.npz')
ss=StandardScaler()
X_full = ss.fit_transform(data["X"])
#X_full = data["X"]
Y_full = data["Y"]


In [3]:
from sklearn.model_selection import KFold
def GetKFoldsIndexes(n_indexes, n_splits, random_state=42, shuffle = True):
    # разбивает множество индексов от 0 до n_indexes-1 на n_splits частей 
    # shuffle - перемешивать индексы или нет  
    kf = KFold(n_splits=n_splits, random_state=random_state, shuffle = True)
    inds = range(n_indexes)
    kf.get_n_splits(inds)
    isplit = kf.get_n_splits(inds)
    folds = []
    for i, ids in enumerate(kf.split(inds)):
        folds.append(ids[1])
    return folds    

In [63]:
def TrainLinearSVM(X, Y, ker = 'linear'):
    t_start = time.time()
    model = SVC(kernel=ker)
    model.fit(X, Y)
    res = {}
    res['a'] = model.coef_
    res['b'] = model.intercept_
    #res['nSV'] = model.
    t_end = time.time()
    res['time'] = t_end - t_start
    return res

def TrainLinearSVM_v2(X, Y, model):
    t_start = time.time()
    model.fit(X, Y)
    res = {}
    res['a'] = model.coef_
    res['b'] = model.intercept_
    #res['nSV'] = model.
    t_end = time.time()
    res['time'] = t_end - t_start
    return res
    
def TestLinearSVM_sub(X,Y, inds, model, prn = False):
    # Тестирование линейного svm в подпространстве inds
    # X, Y - полный набор данных 
    # inds - индексы используемых признаков
    # model['a'] - направляющий вектор в пространстве model['inds'] признаков 
    # model['b'] - смещение 
    t_start = time.time()
    scores = np.dot(X[:,inds], np.transpose(model['a'])[:,0])+model['b']
    miscl_neg = np.sum((Y == -1) & (scores > 0))
    miscl_pos = np.sum((Y == +1) & (scores < 0))
    #print(miscl_neg, miscl_pos)
    acc = 1 - (miscl_neg + miscl_pos) / len(Y)
    t_end = time.time()
    return scores, miscl_pos, miscl_neg, acc, t_end-t_start
    
def TrainLinearSVMs_sub_v2(folds, X, Y, model, prn=False):
    tr_folds = []
    nfolds = len(folds)
    tr_time = 0
    for i in range(nfolds): 
        temp = TrainLinearSVM_v2(X[:,folds[i]], Y, model)
        temp['inds'] = np.array(folds[i])
        tr_folds.append(temp)
        if prn:
            print('sub ', i, ': time=', temp['time']) 
        tr_time = tr_time + temp['time']
        #if prn:
        #    for i in range(nfolds): 
        #        print(i, tr_folds[i])
    return tr_folds, tr_time

def CombLinearSVMs(tr_res):
# tr_res - список результатов обучения на подпространствах
# tr_res[i]['inds'] - индексы признаков, вошедших в подпространство
# tr_res[i]['a'] - направляющий вектор 
# tr_res[i]['b'] - смещение
    time_start = time.time()
    nfolds = len(tr_res)
    a_comb = tr_res[0]['a']
    b_comb = tr_res[0]['b']
    inds_comb = tr_res[0]['inds']
    for i in range(1, nfolds):
        a_comb = np.hstack((a_comb, tr_res[i]['a']))
        b_comb = b_comb + tr_res[i]['b']
        inds_comb = np.hstack((inds_comb, tr_res[i]['inds'])) 
    b_comb = b_comb/nfolds    
    comb = {}
    comb['a'] = a_comb
    comb['b'] = b_comb
    comb['inds'] = inds_comb
    time_end = time.time()
    comb['time'] = time_end - time_start
    return comb 

def TrainAndTestLinearSVM_AllSub(Xtr, Ytr, Xtst, Ytst, reps, prn=True):
    tr_res = []
    nrep = len(reps)
    for j in range(nrep):
        temp = {}
        temp['sub'], tr_time_subs = TrainLinearSVMs_sub(reps[j], Xtr, Ytr, ker='linear', prn = False) 
        tr_res.append(temp)
    #print('ok')
    for j in range(nrep):
        print(f" ---- rep { j+1} --------")
        for i in range(len(reps[j])):
            _,_,_,acc, tst_time = TestLinearSVM_sub(Xtr,Ytr, tr_res[j]['sub'][i]['inds'], tr_res[j]['sub'][i], prn = False)
            tr_res[j]['sub'][i]['acc_tr'] = acc
            tr_res[j]['sub'][i]['tst_time_tr'] = tst_time
            _,_,_,acc, tst_time = TestLinearSVM_sub(Xtst,Ytst, tr_res[j]['sub'][i]['inds'], tr_res[j]['sub'][i], prn = False)
            tr_res[j]['sub'][i]['acc_tst'] = acc
            tr_res[j]['sub'][i]['tst_time_tst'] = tst_time
            if prn:
                # печать с индексами
                #print(f"i={i}, inds={tr_res[j]['sub'][i]['inds']} tr_time={tr_res[j]['sub'][i]['tr_time]:.4f} acc_tr={tr_res[j]['sub'][i]['acc_tr']:.4f}, acc_tst={tr_res[j]['sub'][i]['acc_tst']:.4f}")
                # печать без индексов
                print(f"subspace {i}:  tr_time={tr_res[j]['sub'][i]['time']:.4f} acc_tr={tr_res[j]['sub'][i]['acc_tr']:.4f}, acc_tst= {tr_res[j]['sub'][i]['acc_tst']:.4f}")
                print(f"tst_time_tr: {tr_res[j]['sub'][i]['tst_time_tr']:.4f}, tst_time_tst: {tr_res[j]['sub'][i]['tst_time_tst']:.4f}")

        tr_res[j]['comb'] = CombLinearSVMs(tr_res[j]['sub'])
        tr_res[j]['comb']['tr_time'] = tr_time_subs
        tr_res[j]['comb']['full_time'] = tr_res[j]['comb']['time'] + tr_time_subs

        _, _, _, acc_tr, tst_time_tr = TestLinearSVM_sub(Xtr,Ytr, tr_res[j]['comb']['inds'], tr_res[j]['comb'], prn = False)
        tr_res[j]['comb']['acc_tr'] = acc_tr
        tr_res[j]['comb']['acc_tst_time_tr'] = tst_time_tr 

        _, _, _, acc_tst, tst_time_tst = TestLinearSVM_sub(Xtst,Ytst, tr_res[j]['comb']['inds'], tr_res[j]['comb'], prn = False)
        tr_res[j]['comb']['acc_tst'] = acc_tst
        tr_res[j]['comb']['acc_tst_time_tst'] = tst_time_tst
        if prn:
            print(f"Комбинированная гиперплоскость:  время обучения: {tr_res[j]['comb']['tr_time']:.4f}, время объединения: {tr_res[j]['comb']['time']:.4f}, полное время: {tr_res[j]['comb']['full_time']:.4f} ")
            print(f"acc_tr_comb={tr_res[j]['comb']['acc_tr']:.4f} acc_tst_comb={tr_res[j]['comb']['acc_tst']:.4f}")
            #print(f"comb: inds={tr_res[j]['comb']['inds']} acc_tr_comb={tr_res[j]['comb']['acc_tr']:.4f} acc_tst_comb={tr_res[j]['comb']['acc_tst']:.4f}")
            
            # время тестирования: 
            #print(f"tst_time_tr: {tr_res[j]['sub'][i]['tst_time_tr']:.4f}, tst_time_tst: {tr_res[j]['sub'][i]['tst_time_tst']:.4f}")
    return tr_res

def TrainAndTestLinearSVM_AllSub_v2(Xtr, Ytr, Xtst, Ytst, reps, model, prn=True):
    tr_res = []
    nrep = len(reps)
    for j in range(nrep):
        temp = {}
        temp['sub'], tr_time_subs = TrainLinearSVMs_sub_v2(reps[j], Xtr, Ytr, model, prn = False) 
        tr_res.append(temp)
    #print('ok')
    for j in range(nrep):
        print(f" ---- rep { j+1} --------")
        for i in range(len(reps[j])):
            _,_,_,acc, tst_time = TestLinearSVM_sub(Xtr,Ytr, tr_res[j]['sub'][i]['inds'], tr_res[j]['sub'][i], prn = False)
            tr_res[j]['sub'][i]['acc_tr'] = acc
            tr_res[j]['sub'][i]['tst_time_tr'] = tst_time
            _,_,_,acc, tst_time = TestLinearSVM_sub(Xtst,Ytst, tr_res[j]['sub'][i]['inds'], tr_res[j]['sub'][i], prn = False)
            tr_res[j]['sub'][i]['acc_tst'] = acc
            tr_res[j]['sub'][i]['tst_time_tst'] = tst_time
            if prn:
                # печать с индексами
                #print(f"i={i}, inds={tr_res[j]['sub'][i]['inds']} tr_time={tr_res[j]['sub'][i]['tr_time]:.4f} acc_tr={tr_res[j]['sub'][i]['acc_tr']:.4f}, acc_tst={tr_res[j]['sub'][i]['acc_tst']:.4f}")
                # печать без индексов
                print(f"subspace {i}:  tr_time={tr_res[j]['sub'][i]['time']:.4f} acc_tr={tr_res[j]['sub'][i]['acc_tr']:.4f}, acc_tst= {tr_res[j]['sub'][i]['acc_tst']:.4f}")
                print(f"tst_time_tr: {tr_res[j]['sub'][i]['tst_time_tr']:.4f}, tst_time_tst: {tr_res[j]['sub'][i]['tst_time_tst']:.4f}")

        tr_res[j]['comb'] = CombLinearSVMs(tr_res[j]['sub'])
        tr_res[j]['comb']['tr_time'] = tr_time_subs
        tr_res[j]['comb']['full_time'] = tr_res[j]['comb']['time'] + tr_time_subs

        _, _, _, acc_tr, tst_time_tr = TestLinearSVM_sub(Xtr,Ytr, tr_res[j]['comb']['inds'], tr_res[j]['comb'], prn = False)
        tr_res[j]['comb']['acc_tr'] = acc_tr
        tr_res[j]['comb']['acc_tst_time_tr'] = tst_time_tr 

        _, _, _, acc_tst, tst_time_tst = TestLinearSVM_sub(Xtst,Ytst, tr_res[j]['comb']['inds'], tr_res[j]['comb'], prn = False)
        tr_res[j]['comb']['acc_tst'] = acc_tst
        tr_res[j]['comb']['acc_tst_time_tst'] = tst_time_tst
        if prn:
            print(f"Комбинированная гиперплоскость:  время обучения: {tr_res[j]['comb']['tr_time']:.4f}, время объединения: {tr_res[j]['comb']['time']:.4f}, полное время: {tr_res[j]['comb']['full_time']:.4f} ")
            print(f"acc_tr_comb={tr_res[j]['comb']['acc_tr']:.4f} acc_tst_comb={tr_res[j]['comb']['acc_tst']:.4f}")
            #print(f"comb: inds={tr_res[j]['comb']['inds']} acc_tr_comb={tr_res[j]['comb']['acc_tr']:.4f} acc_tst_comb={tr_res[j]['comb']['acc_tst']:.4f}")
            
            # время тестирования: 
            #print(f"tst_time_tr: {tr_res[j]['sub'][i]['tst_time_tr']:.4f}, tst_time_tst: {tr_res[j]['sub'][i]['tst_time_tst']:.4f}")
    return tr_res



In [60]:
X_train5k,X_test5k,Y_train5k,Y_test5k =train_test_split(X_full,Y_full,train_size = 5000, test_size=5000, random_state=42)
X_train10k, X_test10k,Y_train10k,Y_test10k =train_test_split(X_full,Y_full,train_size = 10000, test_size=10000, random_state=42)
X_train90k, X_test10k,Y_train90k,Y_test10k =train_test_split(X_full,Y_full,train_size = 90000, test_size=10000, random_state=42)

In [8]:
%%time
# Обучение и распознавание по всем признакам для 5к объектов
Xtr = X_train5k 
Ytr = Y_train5k 
Xtst = X_test5k
Ytst = Y_test5k 
n_obj, n_feat = np.shape(Xtr)
print('n_obj = ', n_obj, ' n_feat=', n_feat)
model = TrainLinearSVM(Xtr, Ytr, ker = 'linear') 
inds = np.array(range(n_feat))
_, _, _, acc_tr, tst_time_tr = TestLinearSVM_sub(Xtr,Ytr, inds, model, prn = False)
_, _, _, acc_tst, tst_time_tst = TestLinearSVM_sub(Xtst,Ytst, inds, model, prn = False)

print('acc_tr = ', acc_tr, 'acc_tst=', acc_tst, 'train_time = ', model['time'], ' tst_time_tr =', tst_time_tr, ' tst_time_tst =', tst_time_tst ) 


n_obj =  5000  n_feat= 1000
acc_tr =  1.0 acc_tst= 0.8596 train_time =  8.894100666046143  tst_time_tr = 0.036153554916381836  tst_time_tst = 0.04698967933654785
CPU times: total: 8.78 s
Wall time: 8.98 s


In [9]:
%%time
# Обучение и распознавание по всем признакам для 10к объектов
Xtr = X_train10k 
Ytr = Y_train10k 
Xtst = X_test5k
Ytst = Y_test5k 
n_obj, n_feat = np.shape(Xtr)
print('n_obj = ', n_obj, ' n_feat=', n_feat)
model = TrainLinearSVM(Xtr, Ytr, ker = 'linear') 
inds = np.array(range(n_feat))
_, _, _, acc_tr, tst_time_tr = TestLinearSVM_sub(Xtr,Ytr, inds, model, prn = False)
_, _, _, acc_tst, tst_time_tst = TestLinearSVM_sub(Xtst,Ytst, inds, model, prn = False)

print('acc_tr = ', acc_tr, 'acc_tst=', acc_tst, 'train_time = ', model['time'], ' tst_time_tr =', tst_time_tr, ' tst_time_tst =', tst_time_tst ) 


n_obj =  10000  n_feat= 1000
acc_tr =  0.9729 acc_tst= 0.9368 train_time =  391.1051776409149  tst_time_tr = 0.0895390510559082  tst_time_tst = 0.048734426498413086
CPU times: total: 6min 23s
Wall time: 6min 31s


In [33]:
# комбинирование гиперплоскостей, построенных на подпространствах
Xtr = X_train5k 
Ytr = Y_train5k 
Xtst = X_test5k
Ytst = Y_test5k 
n_splits = 10 # число случайных непересекающихся подпространств
nObj, nFeats = np.shape(Xtr) 
nrep = 2 # число повторов случайных разбиений на подпространствах 
# в данном случае, фактически, не нужно - для используемых наборов данных решение оказалось стабильным без усреднения
# но, возможно, потребуется при увеличении доли неинформативных признаков 
reps = []
for j in range(nrep):
    #state = int(time.time()*10000%1000+j*10)
    #state = j*10
    folds = GetKFoldsIndexes(nFeats, n_splits=n_splits, random_state=j*10, shuffle = False)
    reps.append(folds)
tr_res = TrainAndTestLinearSVM_AllSub_v2(Xtr, Ytr, Xtst, Ytst, reps, prn = True)


 ---- rep 1 --------
subspace 0:  tr_time=6.3523 acc_tr=0.6602, acc_tst= 0.6374
tst_time_tr: 0.0050, tst_time_tst: 0.0040
subspace 1:  tr_time=6.3253 acc_tr=0.6616, acc_tst= 0.6256
tst_time_tr: 0.0050, tst_time_tst: 0.0040
subspace 2:  tr_time=6.5856 acc_tr=0.6588, acc_tst= 0.6254
tst_time_tr: 0.0058, tst_time_tst: 0.0050
subspace 3:  tr_time=6.9870 acc_tr=0.6546, acc_tst= 0.6458
tst_time_tr: 0.0040, tst_time_tst: 0.0040
subspace 4:  tr_time=10.0104 acc_tr=0.6562, acc_tst= 0.6276
tst_time_tr: 0.0040, tst_time_tst: 0.0040
subspace 5:  tr_time=9.5316 acc_tr=0.6456, acc_tst= 0.6198
tst_time_tr: 0.0040, tst_time_tst: 0.0043
subspace 6:  tr_time=9.2725 acc_tr=0.6568, acc_tst= 0.6254
tst_time_tr: 0.0042, tst_time_tst: 0.0050
subspace 7:  tr_time=8.7263 acc_tr=0.6768, acc_tst= 0.6364
tst_time_tr: 0.0044, tst_time_tst: 0.0038
subspace 8:  tr_time=9.5868 acc_tr=0.6488, acc_tst= 0.6296
tst_time_tr: 0.0039, tst_time_tst: 0.0048
subspace 9:  tr_time=10.8707 acc_tr=0.6494, acc_tst= 0.6366
tst_time_

In [57]:
# комбинирование гиперплоскостей, построенных на подпространствах
Xtr = X_train5k 
Ytr = Y_train5k 
Xtst = X_test5k
Ytst = Y_test5k 
n_splits = 10 # число случайных непересекающихся подпространств
nObj, nFeats = np.shape(Xtr) 
nrep = 2 # число повторов случайных разбиений на подпространствах 
# в данном случае, фактически, не нужно - для используемых наборов данных решение оказалось стабильным без усреднения
# но, возможно, потребуется при увеличении доли неинформативных признаков 
reps = []
for j in range(nrep):
    #state = int(time.time()*10000%1000+j*10)
    #state = j*10
    folds = GetKFoldsIndexes(nFeats, n_splits=n_splits, random_state=j*10, shuffle = False)
    reps.append(folds)
model = LinearSVC(C=0.01, penalty="l1", dual=False)
tr_res = TrainAndTestLinearSVM_AllSub_v2(Xtr, Ytr, Xtst, Ytst, reps, model, prn = True)

 ---- rep 1 --------
subspace 0:  tr_time=0.0320 acc_tr=0.6556, acc_tst= 0.6368
tst_time_tr: 0.0050, tst_time_tst: 0.0060
subspace 1:  tr_time=0.0180 acc_tr=0.6598, acc_tst= 0.6222
tst_time_tr: 0.0040, tst_time_tst: 0.0040
subspace 2:  tr_time=0.0190 acc_tr=0.6554, acc_tst= 0.6238
tst_time_tr: 0.0040, tst_time_tst: 0.0050
subspace 3:  tr_time=0.0187 acc_tr=0.6442, acc_tst= 0.6498
tst_time_tr: 0.0040, tst_time_tst: 0.0040
subspace 4:  tr_time=0.0207 acc_tr=0.6518, acc_tst= 0.6230
tst_time_tr: 0.0050, tst_time_tst: 0.0040
subspace 5:  tr_time=0.0209 acc_tr=0.6448, acc_tst= 0.6186
tst_time_tr: 0.0045, tst_time_tst: 0.0040
subspace 6:  tr_time=0.0274 acc_tr=0.6534, acc_tst= 0.6200
tst_time_tr: 0.0050, tst_time_tst: 0.0040
subspace 7:  tr_time=0.0222 acc_tr=0.6726, acc_tst= 0.6340
tst_time_tr: 0.0040, tst_time_tst: 0.0050
subspace 8:  tr_time=0.0190 acc_tr=0.6446, acc_tst= 0.6320
tst_time_tr: 0.0040, tst_time_tst: 0.0040
subspace 9:  tr_time=0.0172 acc_tr=0.6444, acc_tst= 0.6342
tst_time_tr

In [64]:
# комбинирование гиперплоскостей, построенных на подпространствах
Xtr = X_train90k 
Ytr = Y_train90k 
Xtst = X_test10k
Ytst = Y_test10k 
n_splits = 10 # число случайных непересекающихся подпространств
nObj, nFeats = np.shape(Xtr) 
nrep = 2 # число повторов случайных разбиений на подпространствах 
# в данном случае, фактически, не нужно - для используемых наборов данных решение оказалось стабильным без усреднения
# но, возможно, потребуется при увеличении доли неинформативных признаков 
reps = []
for j in range(nrep):
    #state = int(time.time()*10000%1000+j*10)
    #state = j*10
    folds = GetKFoldsIndexes(nFeats, n_splits=n_splits, random_state=j*10, shuffle = False)
    reps.append(folds)
model = LinearSVC(C=0.01, penalty="l1", dual=False)
tr_res = TrainAndTestLinearSVM_AllSub_v2(Xtr, Ytr, Xtst, Ytst, reps, model, prn = True)

 ---- rep 1 --------
subspace 0:  tr_time=0.2966 acc_tr=0.6484, acc_tst= 0.6492
tst_time_tr: 0.0993, tst_time_tst: 0.0110
subspace 1:  tr_time=0.3244 acc_tr=0.6462, acc_tst= 0.6491
tst_time_tr: 0.0987, tst_time_tst: 0.0091
subspace 2:  tr_time=0.3296 acc_tr=0.6441, acc_tst= 0.6381
tst_time_tr: 0.0975, tst_time_tst: 0.0119
subspace 3:  tr_time=0.3054 acc_tr=0.6542, acc_tst= 0.6487
tst_time_tr: 0.0975, tst_time_tst: 0.0169
subspace 4:  tr_time=0.3218 acc_tr=0.6401, acc_tst= 0.6380
tst_time_tr: 0.1191, tst_time_tst: 0.0067
subspace 5:  tr_time=0.3292 acc_tr=0.6362, acc_tst= 0.6347
tst_time_tr: 0.1936, tst_time_tst: 0.0236
subspace 6:  tr_time=0.3194 acc_tr=0.6490, acc_tst= 0.6465
tst_time_tr: 0.1177, tst_time_tst: 0.0168
subspace 7:  tr_time=0.2959 acc_tr=0.6523, acc_tst= 0.6565
tst_time_tr: 0.1141, tst_time_tst: 0.0100
subspace 8:  tr_time=0.2990 acc_tr=0.6453, acc_tst= 0.6388
tst_time_tr: 0.0974, tst_time_tst: 0.0100
subspace 9:  tr_time=0.3337 acc_tr=0.6489, acc_tst= 0.6446
tst_time_tr

In [44]:
def RFEselection(X_train, Y_train, nf_sel):
#Feature ranking with recursive feature elimination
#The number of features to select. If None, half of the features are selected. 
#If integer, the parameter is the absolute number of features to select. 
#If float between 0 and 1, it is the fraction of features to select
    X_train_df=pd.DataFrame(X_train)
    #from sklearn.feature_selection import RFE
    svc_lin=SVC(kernel='linear')
    svm_rfe_model=RFE(estimator=svc_lin, n_features_to_select = nf_sel, step = 1)
    svm_rfe_model_fit=svm_rfe_model.fit(X_train_df,Y_train)
    feat_index = pd.Series(data = svm_rfe_model_fit.ranking_)
    signi_feat_rfe = feat_index[feat_index==1].index # indexes of selected features
    #print('Significant features from RFE',signi_feat_rfe)
    #print(svm_rfe_model_fit.ranking_)
    #print('score = ', svm_rfe_model_fit.score(X_trains_df, Y_train))
    sel_feat = svm_rfe_model_fit.transform(X_train_df) # submatrix of selected features
    #score = svm_rfe_model_fit.score(X_train_df, Y_train)  
    return signi_feat_rfe, feat_index, svm_rfe_model_fit.ranking_, sel_feat 

def RFEselection_and_find_acc(X_trains, Y_train, X_vals, Y_val, X_tests, Y_test, n_sel_feat):
    time_start = time.time()
    selfeat_inds, feat_ranks, ranking, X_selfeat = RFEselection(X_trains, Y_train, n_sel_feat)
    svc_selfeat=SVC(kernel='linear')
    svc_selfeat.fit(X_selfeat,Y_train)
    time_end_train = time.time() 
    print('train_time =', time_end_train-time_start)
    y_pred_train = svc_selfeat.predict(X_selfeat)
    acc_tr = accuracy_score(Y_train, y_pred_train)
    #print(selfeat_inds)
    X_selfeat_test = X_tests[:,selfeat_inds]
    y_pred_test = svc_selfeat.predict(X_selfeat_test)
    acc_tst = accuracy_score(Y_test, y_pred_test)

    X_selfeat_val = X_vals[:,selfeat_inds]
    y_pred_val = svc_selfeat.predict(X_selfeat_val)
    acc_val = accuracy_score(Y_val, y_pred_val)

    print('n_sel_feat=',n_sel_feat, 'selfeat_inds:',selfeat_inds)
    print('accuracy train:', acc_tr, 'accuracy test:', acc_tst, 'accuracy val:', acc_val)
    return selfeat_inds, acc_tr, acc_tst, acc_val, time_end_train-time_start

In [45]:
Xtr = X_train5k 
Ytr = Y_train5k 
Xtst = X_test5k
Ytst = Y_test5k 
n_sel_feat = 0.8
RFEselection_and_find_acc(Xtr, Ytr, Xtst, Ytst, Xtst, Ytst, n_sel_feat)

train_time = 1634.2938168048859
n_sel_feat= 0.8 selfeat_inds: Index([  1,   2,   3,   6,   8,   9,  10,  11,  12,  13,
       ...
       989, 990, 991, 992, 994, 995, 996, 997, 998, 999],
      dtype='int64', length=800)
accuracy train: 1.0 accuracy test: 0.8558 accuracy val: 0.8558


(Index([  1,   2,   3,   6,   8,   9,  10,  11,  12,  13,
        ...
        989, 990, 991, 992, 994, 995, 996, 997, 998, 999],
       dtype='int64', length=800),
 1.0,
 0.8558,
 0.8558,
 1634.2938168048859)

In [54]:
%%time
# l1-based feature selection
#Linear models penalized with the L1 norm have sparse solutions: many of their estimated coefficients are zero. 
# When the goal is to reduce the dimensionality of the data to use with another classifier, they can be used along with SelectFromModel 
#to select the non-zero coefficients. In particular, sparse estimators useful for this purpose are the Lasso for regression, 
#and of LogisticRegression and LinearSVC for classification
# With SVMs and logistic regression, the parameter C controls the sparsity: the smaller C the fewer features selected
Xtr = X_train5k 
Ytr = Y_train5k 
Xtst = X_test5k
Ytst = Y_test5k 
n_obj, n_feat = np.shape(Xtr)
print('n_obj = ', n_obj, ' n_feat=', n_feat)
model = LinearSVC(C=0.01, penalty="l1", dual=False)
model = TrainLinearSVM_v2(Xtr, Ytr, model) 

inds = np.array(range(n_feat))
_, _, _, acc_tr, tst_time_tr = TestLinearSVM_sub(Xtr,Ytr, inds, model, prn = False)
_, _, _, acc_tst, tst_time_tst = TestLinearSVM_sub(Xtst,Ytst, inds, model, prn = False)

print('acc_tr = ', acc_tr, 'acc_tst=', acc_tst, 'train_time = ', model['time'], ' tst_time_tr =', tst_time_tr, ' tst_time_tst =', tst_time_tst ) 

n_obj =  5000  n_feat= 1000
acc_tr =  0.9558 acc_tst= 0.9458 train_time =  0.37770938873291016  tst_time_tr = 0.04251360893249512  tst_time_tst = 0.04299592971801758
CPU times: total: 438 ms
Wall time: 463 ms


In [55]:
%%time
# l1-based feature selection
#Linear models penalized with the L1 norm have sparse solutions: many of their estimated coefficients are zero. 
# When the goal is to reduce the dimensionality of the data to use with another classifier, they can be used along with SelectFromModel 
#to select the non-zero coefficients. In particular, sparse estimators useful for this purpose are the Lasso for regression, 
#and of LogisticRegression and LinearSVC for classification
# With SVMs and logistic regression, the parameter C controls the sparsity: the smaller C the fewer features selected
Xtr = X_train10k 
Ytr = Y_train10k 
Xtst = X_test5k
Ytst = Y_test5k 
n_obj, n_feat = np.shape(Xtr)
print('n_obj = ', n_obj, ' n_feat=', n_feat)
model = LinearSVC(C=0.01, penalty="l1", dual=False)
model = TrainLinearSVM_v2(Xtr, Ytr, model) 

inds = np.array(range(n_feat))
_, _, _, acc_tr, tst_time_tr = TestLinearSVM_sub(Xtr,Ytr, inds, model, prn = False)
_, _, _, acc_tst, tst_time_tst = TestLinearSVM_sub(Xtst,Ytst, inds, model, prn = False)

print('acc_tr = ', acc_tr, 'acc_tst=', acc_tst, 'train_time = ', model['time'], ' tst_time_tr =', tst_time_tr, ' tst_time_tst =', tst_time_tst ) 

n_obj =  10000  n_feat= 1000
acc_tr =  0.9577 acc_tst= 0.9494 train_time =  1.0065150260925293  tst_time_tr = 0.11534404754638672  tst_time_tst = 0.04607057571411133
CPU times: total: 1.14 s
Wall time: 1.17 s


In [61]:
%%time
# l1-based feature selection
#Linear models penalized with the L1 norm have sparse solutions: many of their estimated coefficients are zero. 
# When the goal is to reduce the dimensionality of the data to use with another classifier, they can be used along with SelectFromModel 
#to select the non-zero coefficients. In particular, sparse estimators useful for this purpose are the Lasso for regression, 
#and of LogisticRegression and LinearSVC for classification
# With SVMs and logistic regression, the parameter C controls the sparsity: the smaller C the fewer features selected
Xtr = X_train90k 
Ytr = Y_train90k 
Xtst = X_test10k
Ytst = Y_test10k 
n_obj, n_feat = np.shape(Xtr)
print('n_obj = ', n_obj, ' n_feat=', n_feat)
model = LinearSVC(C=0.01, penalty="l1", dual=False)
model = TrainLinearSVM_v2(Xtr, Ytr, model) 

inds = np.array(range(n_feat))
_, _, _, acc_tr, tst_time_tr = TestLinearSVM_sub(Xtr,Ytr, inds, model, prn = False)
_, _, _, acc_tst, tst_time_tst = TestLinearSVM_sub(Xtst,Ytst, inds, model, prn = False)

print('acc_tr = ', acc_tr, 'acc_tst=', acc_tst, 'train_time = ', model['time'], ' tst_time_tr =', tst_time_tr, ' tst_time_tst =', tst_time_tst ) 

n_obj =  90000  n_feat= 1000
acc_tr =  0.9504444444444444 acc_tst= 0.9467 train_time =  6.845935583114624  tst_time_tr = 1.0006425380706787  tst_time_tst = 0.09502530097961426
CPU times: total: 8 s
Wall time: 7.94 s


In [ ]:
# предварительный отбор признаков 

In [ ]:
#Random Subspace Ensemble via Bagging
#https://machinelearningmastery.com/random-subspace-ensemble-with-python/ 

In [ ]:
## Initialize the SVM with class_weight='balanced'
svm_model = SVC(class_weight='balanced', random_state=42)
# Method 2: Manual Class Weighting 
# Manually defining weights (e.g., class 0 has weight 1, class 1 has weight 10)
weights = {0: 1, 1: 10}

svm_model_manual = SVC(class_weight=weights, random_state=42)
svm_model_manual.fit(X_train, y_train)

In [ ]:
#This example will also work by replacing SVC(kernel="linear") with SGDClassifier(loss="hinge"). Setting the loss parameter of the SGDClassifier equal to hinge will yield behaviour such as that of an SVC with a linear kernel.

#For example try instead of the SVC:
clf = SGDClassifier(n_iter=100, alpha=0.01) 


In [ ]:
#SVM-Anova: SVM with univariate feature selection 
#https://scikit-learn.org/stable/auto_examples/svm/plot_svm_anova.html 

In [ ]:
# Univariate Feature Selection 
#https://scikit-learn.org/stable/auto_examples/feature_selection/plot_feature_selection.html